In [ ]:
# =============================================================
# CropResQ - Standardized DINOv2 Pipeline + ROC & PDF Report
# =============================================================
!pip install -q transformers reportlab seaborn matplotlib scikit-learn
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from transformers import AutoImageProcessor, AutoModel
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support, roc_curve, auc
from sklearn.preprocessing import label_binarize

from reportlab.lib.pagesizes import letter
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image as RLImage, Table, TableStyle, PageBreak
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle

SEED = 42
BATCH_SIZE = 32
NUM_EPOCHS = 25
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 1e-2
IMG_SIZE = 224
MODEL_TAG = "DINOv2"
HF_MODEL_NAME = "facebook/dinov2-base"
OUTPUT_DIR = "/kaggle/working/dinov2_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

candidate_paths = ["/kaggle/input/datasets/varun2ks05/newdata", "/kaggle/input/newdata"]
DATASET_DIR = None
for p in candidate_paths:
    if os.path.exists(os.path.join(p, "train")):
        DATASET_DIR = p
        break
if DATASET_DIR is None:
    matches = glob.glob("/kaggle/input/**/train", recursive=True)
    if matches: DATASET_DIR = os.path.dirname(matches[0])
    else: raise FileNotFoundError("Dataset train/val/test splits not found.")

processor = AutoImageProcessor.from_pretrained(HF_MODEL_NAME)
norm_mean = processor.image_mean if hasattr(processor, "image_mean") else [0.485, 0.456, 0.406]
norm_std = processor.image_std if hasattr(processor, "image_std") else [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=norm_mean, std=norm_std)
])
eval_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=norm_mean, std=norm_std)
])

val_subdir = "val" if os.path.exists(os.path.join(DATASET_DIR, "val")) else "validation"
train_dataset = datasets.ImageFolder(os.path.join(DATASET_DIR, "train"), transform=train_transforms)
val_dataset   = datasets.ImageFolder(os.path.join(DATASET_DIR, val_subdir), transform=eval_transforms)
test_dataset  = datasets.ImageFolder(os.path.join(DATASET_DIR, "test"), transform=eval_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

class_names = train_dataset.classes
NUM_CLASSES = len(class_names)

class DINOv2Classifier(nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden_dim = self.backbone.config.hidden_size
        self.classifier = nn.Sequential(nn.LayerNorm(hidden_dim), nn.Linear(hidden_dim, num_classes))
    def forward(self, pixel_values):
        outputs = self.backbone(pixel_values=pixel_values)
        return self.classifier(outputs.last_hidden_state[:, 0, :])

model = DINOv2Classifier(HF_MODEL_NAME, NUM_CLASSES).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

best_val_acc = 0.0
best_model_path = os.path.join(OUTPUT_DIR, f"best_{MODEL_TAG.lower()}_rice_model.pth")
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss, correct_train, total_train = 0.0, 0, 0
    for images, targets in train_loader:
        images, targets = images.to(DEVICE, non_blocking=True), targets.to(DEVICE, non_blocking=True)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        correct_train += (outputs.argmax(1) == targets).sum().item()
        total_train += targets.size(0)
    scheduler.step()

    model.eval()
    val_loss, correct_val, total_val = 0.0, 0, 0
    with torch.no_grad():
        for images, targets in val_loader:
            images, targets = images.to(DEVICE, non_blocking=True), targets.to(DEVICE, non_blocking=True)
            outputs = model(images)
            val_loss += criterion(outputs, targets).item() * images.size(0)
            correct_val += (outputs.argmax(1) == targets).sum().item()
            total_val += targets.size(0)

    epoch_train_loss, epoch_train_acc = running_loss / total_train, (correct_train / total_train) * 100
    epoch_val_loss, epoch_val_acc = val_loss / total_val, (correct_val / total_val) * 100
    history["train_loss"].append(epoch_train_loss); history["val_loss"].append(epoch_val_loss)
    history["train_acc"].append(epoch_train_acc); history["val_acc"].append(epoch_val_acc)

    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        torch.save(model.state_dict(), best_model_path)
    print(f"[{MODEL_TAG}] Epoch [{epoch+1:02d}/{NUM_EPOCHS:02d}] Train Acc: {epoch_train_acc:.2f}% | Val Acc: {epoch_val_acc:.2f}%")

model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
model.eval()
all_preds, all_targets, all_probs = [], [], []
with torch.no_grad():
    for images, targets in test_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_targets.extend(targets.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds, all_targets, all_probs = np.array(all_preds), np.array(all_targets), np.array(all_probs)
np.save(os.path.join(OUTPUT_DIR, f"{MODEL_TAG.lower()}_test_probs.npy"), all_probs)

overall_acc = accuracy_score(all_targets, all_preds) * 100
macro_prec, macro_rec, macro_f1, _ = precision_recall_fscore_support(all_targets, all_preds, average="macro", zero_division=0)
cm = confusion_matrix(all_targets, all_preds)
class_accuracies = (cm.diagonal() / cm.sum(axis=1)) * 100
rep_dict = classification_report(all_targets, all_preds, target_names=class_names, digits=4, zero_division=0, output_dict=True)

curves_path = os.path.join(OUTPUT_DIR, "training_curves.png")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.2))
ax1.plot(range(1, NUM_EPOCHS + 1), history["train_loss"], label="Train Loss")
ax1.plot(range(1, NUM_EPOCHS + 1), history["val_loss"], label="Val Loss")
ax1.set_title(f"{MODEL_TAG} Loss Curves"); ax1.legend()
ax2.plot(range(1, NUM_EPOCHS + 1), history["train_acc"], label="Train Acc")
ax2.plot(range(1, NUM_EPOCHS + 1), history["val_acc"], label="Val Acc")
ax2.set_title(f"{MODEL_TAG} Accuracy Curves"); ax2.legend()
plt.tight_layout(); plt.savefig(curves_path, dpi=300); plt.close()

cm_path = os.path.join(OUTPUT_DIR, "confusion_matrix.png")
plt.figure(figsize=(7, 5.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.title(f"{MODEL_TAG} Confusion Matrix (Test Set)")
plt.xticks(rotation=35, ha="right"); plt.tight_layout(); plt.savefig(cm_path, dpi=300); plt.close()

y_test_bin = label_binarize(all_targets, classes=list(range(NUM_CLASSES)))
roc_path = os.path.join(OUTPUT_DIR, "roc_curve.png")
plt.figure(figsize=(8, 5.5))
for i in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], all_probs[:, i])
    plt.plot(fpr, tpr, lw=1.5, label=f"{class_names[i]} (AUC = {auc(fpr, tpr):.3f})")
fpr_m, tpr_m, _ = roc_curve(y_test_bin.ravel(), all_probs.ravel())
plt.plot(fpr_m, tpr_m, color="deeppink", linestyle=":", lw=2, label=f"Micro-avg (AUC = {auc(fpr_m, tpr_m):.3f})")
plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.title(f"{MODEL_TAG} Multiclass ROC Curves"); plt.legend(loc="lower right", fontsize=8)
plt.tight_layout(); plt.savefig(roc_path, dpi=300); plt.close()

REPORT_PDF_PATH = os.path.join(OUTPUT_DIR, f"{MODEL_TAG}_Canonical_V1_Experiment_Report.pdf")
doc = SimpleDocTemplate(REPORT_PDF_PATH, pagesize=letter, leftMargin=36, rightMargin=36, topMargin=36, bottomMargin=36)
styles = getSampleStyleSheet()
title_style = ParagraphStyle('DocTitle', parent=styles['Title'], fontName='Helvetica-Bold', fontSize=18, leading=22, alignment=0)
h2_style = ParagraphStyle('Heading2Custom', fontName='Helvetica-Bold', fontSize=11, leading=14, spaceBefore=8, spaceAfter=4)
body_style = ParagraphStyle('BodyCustom', fontName='Helvetica', fontSize=8.5, leading=11)

def std_table_style():
    return TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor("#2C3E50")),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('FONTSIZE', (0, 0), (-1, -1), 8),
        ('GRID', (0, 0), (-1, -1), 0.5, colors.HexColor("#D1D5DB")),
    ])

story = [
    Paragraph(f"Rice Leaf Disease Classification: {MODEL_TAG}", title_style),
    Spacer(1, 10),
    Paragraph("1. Experiment Configuration", h2_style),
    Table([
        ["Parameter", "Value"], ["Model Backbone", HF_MODEL_NAME],
        ["Image Dimensions", f"{IMG_SIZE} x {IMG_SIZE}"], ["Batch Size", str(BATCH_SIZE)],
        ["Epochs", str(NUM_EPOCHS)], ["Optimizer / LR", f"AdamW / {LEARNING_RATE}"],
        ["Peak Val Accuracy", f"{best_val_acc:.2f}%"], ["Final Test Accuracy", f"{overall_acc:.2f}%"],
        ["Macro F1-Score", f"{macro_f1*100:.2f}%"]
    ], colWidths=[200, 320], style=std_table_style()),
    Spacer(1, 10),
    Paragraph("2. Test Classification Report", h2_style)
]

clf_data = [["Class", "Precision", "Recall", "F1-Score", "Class Acc", "Support"]]
for idx, c in enumerate(class_names):
    clf_data.append([c, f"{rep_dict[c]['precision']:.4f}", f"{rep_dict[c]['recall']:.4f}", f"{rep_dict[c]['f1-score']:.4f}", f"{class_accuracies[idx]:.2f}%", str(int(rep_dict[c]['support']))])
clf_data.append(["Macro Avg", f"{rep_dict['macro avg']['precision']:.4f}", f"{rep_dict['macro avg']['recall']:.4f}", f"{rep_dict['macro avg']['f1-score']:.4f}", f"{overall_acc:.2f}%", str(len(all_targets))])
story.append(Table(clf_data, colWidths=[160, 70, 70, 70, 80, 70], style=std_table_style()))
story.append(Spacer(1, 10))
story.append(Paragraph("3. Learning Curves", h2_style))
story.append(RLImage(curves_path, width=520, height=218))
story.append(PageBreak())
story.append(Paragraph("4. Confusion Matrix", h2_style))
story.append(RLImage(cm_path, width=400, height=314))
story.append(Spacer(1, 10))
story.append(Paragraph("5. Receiver Operating Characteristic (ROC)", h2_style))
story.append(RLImage(roc_path, width=460, height=316))

doc.build(story)
print(f"[{MODEL_TAG}] Processing complete. PDF generated: {REPORT_PDF_PATH}")